<a href="https://colab.research.google.com/github/lawesworks/vision-model-workbench/blob/main/YOLO_OD_Trainer_Enablement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi -q
!nvidia-smi

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# CONFIGURATION

# You shouldn't need to change anything other than the roboflow URL and the sample image URL
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

# Roboflow Project Settings
Roboflow_Project_URL = "hhttps://universe.roboflow.com/crowd-zuf0q/crowd-5ropi"
Roboflow_Dataset_Version = 1

# URL Regex
import re
Roboflow_URL_Regex_Pattern = r"roboflow\.com/([^/]+)/([^/?#]+)"
match = re.search(Roboflow_URL_Regex_Pattern, Roboflow_Project_URL)

#-------------------------------------------------------------------------------

# YOLO Model Config
YOLO_Model_Version = "yolov8"
YOLO_Model_Size = "n"

# Training Hyper-parameter Configs
Config_Epochs = 30
Config_Image_Size = 640
Config_Batch_Size = 16

#-------------------------------------------------------------------------------
# Auto-Derived Parameters
if match:
    Roboflow_Project_Workspace = match.group(1)
    Roboflow_Project_Slug = match.group(2)
else:
    print("Roboflow URL format not recognized")
    Roboflow_Project_Workspace = ""
    Roboflow_Project_Slug = ""

Roboflow_Project_Name = ""
Roboflow_Project_Folder = ""
LATEST_PREDICT_DIR = "runs/detect/predict"
LATEST_TRAIN_DIR   = "runs/detect/train"

#-------------------------------------------------------------------------------

print(f"""
===== Training Configuration =====

Project URL      : {Roboflow_Project_URL}
Workspace        : {Roboflow_Project_Workspace}
Project Slug     : {Roboflow_Project_Slug}
Dataset Version  : {Roboflow_Dataset_Version}

Project Name     : {Roboflow_Project_Name}
Project Folder   : {Roboflow_Project_Folder}


Model            : {YOLO_Model_Version}
Model Size       : {YOLO_Model_Size}
Epochs           : {Config_Epochs}
Image Size       : {Config_Image_Size}
Batch Size       : {Config_Batch_Size}

==================================
""")

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Load Roboflow API Key

# Retrieves the Roboflow API key that you've set in Google Colab's Secrets
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

from google.colab import userdata
api_key = userdata.get('ROBOFLOW_API_KEY')

if api_key is None:
    raise ValueError("ROBOFLOW_API_KEY not found. Check Colab Secrets.")

print("Roboflow API Key Found")

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Load Libraries

# These are commonly used libraries that we will use in this script
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

from IPython.display import Image, display
import glob
import os

print(f"""
======= Imported Libraries =======

Ipython.display  : Image
Ipython.display  : display
glob
os

==================================
""")

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Install Ultralytics

# Ultralytics is necessary because it provides the YOLO framework and APIs required to load the .pt model and run object detection inference in Python.
# This .pt file (weights file) is what we will use in our inference scripts
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

print("Installing Ultralytics (Please wait)\n")

!pip install -q roboflow ultralytics

print("\nCompleted Ultralytics Install")

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Load Libraries

# Commonly used libraries we will use in this project
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

import math
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import Image as DisplayImage, display
import torch
import glob
import os

print(f"""
===== Imported Libraries =====

Ipython.display  : Image
Ipython.display  : DisplayImage
matplotlib       : plt
torch
math
glob
os

==================================
""")

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# HELPER FUNCTIONS

# Reusable functions used in this script to automate repetitive or iterative steps / more to come
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++



# function to create image grid (Left → Right, Row Wrap) of images saved into the Predicted Folder
# ----------------------------------------------------------------------------------------------------
def show_image_grid_paged(image_paths, cols=5, per_page=20, page=1, figsize_per_cell=3):
    """
    Display images in a true grid, paged.
    - cols: images per row
    - per_page: total images per page
    - page: 1-based page index
    - figsize_per_cell: size multiplier per grid cell
    """

    if not image_paths:
        print("No images to display.")
        return

    start = (page - 1) * per_page
    end = min(start + per_page, len(image_paths))
    page_paths = image_paths[start:end]

    rows = math.ceil(len(page_paths) / cols)
    fig_w = cols * figsize_per_cell
    fig_h = rows * figsize_per_cell

    fig, axes = plt.subplots(rows, cols, figsize=(fig_w, fig_h))
    axes = axes.flatten() if isinstance(axes, (list, tuple)) is False else axes

    # If only one subplot, axes may not be iterable the same way
    try:
        axes = axes.flatten()
    except Exception:
        axes = [axes]

    for ax in axes:
        ax.axis("off")

    for ax, img_path in zip(axes, page_paths):
        img = Image.open(img_path).convert("RGB")
        ax.imshow(img)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    print(f"Showing images {start+1}–{end} of {len(image_paths)} (page {page})")



# function to count images in given folder
# ----------------------------------------------------------------------------------------------------
def count_images(folder):
    extensions = ("*.jpg", "*.jpeg", "*.png")
    count = 0
    for ext in extensions:
        count += len(glob.glob(os.path.join(folder, ext)))
    return count



# function to get latest training path DIR
# ----------------------------------------------------------------------------------------------------
def get_latest_training_path():
    training_dirs = sorted(
        glob.glob("/content/runs/detect/train*"),
        key=os.path.getmtime
    )

    if not training_dirs:
        raise FileNotFoundError("No YOLO training runs found in runs/detect/")

    train_dir = training_dirs[-1]
    print(f"\nUsing training run from: {train_dir}")
    return train_dir


# function to get latest prediction path DIR
# ----------------------------------------------------------------------------------------------------
def get_latest_prediction_path():
  predict_dirs = sorted(
    glob.glob("runs/detect/predict*"),
    key=os.path.getmtime
  )

  PREDICT_DIR = predict_dirs[-1]
  print(f"\n\nUsing predictions from: {PREDICT_DIR}")
  return PREDICT_DIR


# function to get latest model best.pt path
# ----------------------------------------------------------------------------------------------------
def get_latest_pt_path(latest_train_path):
  pt_files = glob.glob(f"{latest_train_path}/weights/best.pt", recursive=True)

  if not pt_files:
      raise FileNotFoundError("No best.pt file found")

  return pt_files[0]


# function to get list of predicted / inferenced images
# ----------------------------------------------------------------------------------------------------
def get_inferenced_images(predict_dir):
  image_paths = glob.glob(os.path.join(predict_dir, "*.jpg")) + \
              glob.glob(os.path.join(predict_dir, "*.png"))

  return sorted(image_paths)


print(f"""
===== Loaded Helper Functions =====

show_image_grid_paged              :
count_images                       :
get_latest_training_path           :
get_latest_prediction_path         :
get_latest_pt_path                 :
get_inferenced_images              :

==================================
""")

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Import the Dataset

# Leveraging Roboflow (or even HuggingFace) enables ready-to-use, versioned annotated datasets; This
# simplifies dataset management, preprocessing, and creating reproducible training workflows
# - all without having to staore or manage the data locally
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

print(f"Importing Dataset: [{Roboflow_Project_Slug}, Version: [{Roboflow_Dataset_Version}] from: [{Roboflow_Project_Workspace}] (Please wait)\n")

from roboflow import Roboflow


rf = Roboflow(api_key=api_key )
project = rf.workspace(Roboflow_Project_Workspace).project(Roboflow_Project_Slug)

dataset = project.version(Roboflow_Dataset_Version).download(YOLO_Model_Version)  # adjust version if needed

# Store Official Project Name and Folder location to Local Variables
Roboflow_Project_Name = project.name
Roboflow_Project_Folder = dataset.location

print(f"\nCompleted Dataset Import: {Roboflow_Project_Name}")

# Get Image Count within Training and Validation folders
Data_Train_Count = count_images(os.path.join(Roboflow_Project_Folder, "train", "images"))
Data_Valid_Count = count_images(os.path.join(Roboflow_Project_Folder, "valid", "images"))


print(f"""
===== Training Data Statistics =====

Roboflow Project Name              : {Roboflow_Project_Name}
Roboflow Project Folder            : {Roboflow_Project_Folder}
Training Image Count               : {Data_Train_Count}
Validation Image Count             : {Data_Valid_Count}

====================================
""")

In [ ]:

# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Get location of the yaml file for this project

# The YAML file defines the dataset configuration for training, including paths to the training/validation
# data and the list of class names, so the YOLO training script knows where the images and labels are
# and how to map class IDs to object categories.
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

yaml_files = glob.glob(f"{Roboflow_Project_Folder}/data.yaml", recursive=True)

if not yaml_files:
    raise FileNotFoundError("No data.yaml file found")

DATA_YAML_PATH = yaml_files[0]

print(f"Using dataset config: {DATA_YAML_PATH}")

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Train model on the data
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++


print("Training The Model (Please wait)\n")
print(f"""
===== Training Hyper-parameters =====

Model            : {YOLO_Model_Version}
Model Size       : {YOLO_Model_Size}
Epochs           : {Config_Epochs}
Image Size       : {Config_Image_Size}
Batch Size       : {Config_Batch_Size}
Train Count      : {Data_Train_Count}
Valid Count      : {Data_Valid_Count}

=====================================
""")


from ultralytics import YOLO

DATA_YAML_PATH = DATA_YAML_PATH

model = YOLO("yolov8n.pt")  # small + fast starter model
results = model.train(
    data=DATA_YAML_PATH,
    epochs=Config_Epochs,
    imgsz=Config_Image_Size,
    batch=Config_Batch_Size
)

# get latest training folder - save it to variabble
LATEST_TRAIN_DIR = get_latest_training_path()


In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Get location of the latest weights generated after training
# The .pt weights file stores the trained YOLO model parameters learned during training, allowing the
# model to perform inference or continue training without retraining from scratch.
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

MODEL_PT_PATH = get_latest_pt_path(LATEST_TRAIN_DIR)

print(f"Using PT File: {MODEL_PT_PATH}")

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Run the model against the test data in your project
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

best_model_path = glob.glob(MODEL_PT_PATH)[-1]
model = YOLO(best_model_path)

model.predict(source = Roboflow_Project_Folder + "/test/images", save=True, conf=0.25)

LATEST_PREDICT_DIR = get_latest_prediction_path()

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Get list of latest predicted / inferenced images
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

image_paths = get_inferenced_images(LATEST_PREDICT_DIR)

print(f"Found {len(image_paths)} predicted images")

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Show images in the Recent Predictions Folder
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

#show first 20 images, 5 per row
show_image_grid_paged(image_paths, cols=8, per_page=24, page=1)

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# View the training plots (Loss Function plots / Accuracy)
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

DisplayImage(filename=f'/{LATEST_TRAIN_DIR}/results.png', width=600)

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# View the Confusio Matrix
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

DisplayImage(filename=f'/{LATEST_TRAIN_DIR}/confusion_matrix.png', width=600)

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# View training metrics
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

# model.val(data=DATA_YAML_PATH) # dumps everything (verbose)

metrics = model.val(data=DATA_YAML_PATH)

print("mAP@0.5      :", metrics.box.map50)
print("mAP@0.5:0.95 :", metrics.box.map)
print("Precision    :", metrics.box.mp)
print("Recall       :", metrics.box.mr)


In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Run inference on an arbitrary image and save it to the content folder
#
# alternatively, for images, input could be a URL
# CONFIG_SAMPLE_IMAGE_URL = "https://ufpro.com/storage/app/media/Blog/Rifle%20Fundamentals%20Ep9/thumb/560x415.crop/Header.jpg"
# model.predict(source=f"CONFIG_SAMPLE_IMAGE_URL", save=True, conf=0.25)
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

from google.colab import files

uploaded = files.upload()
uploaded_image_filename = list(uploaded.keys())[0]

input_image = f"/content/{uploaded_image_filename}"

# Save to disk
with open(input_image, "wb") as f:
    f.write(uploaded[uploaded_image_filename])

results = model.predict(
    source=input_image,
    save=True,
    conf=0.1,
    name="content",
    exist_ok=True
)


In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# View the annotated result
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

# find the saved image (YOLO keeps original name or auto-generated one)
# predicted_images = glob.glob(os.path.join("/content/", "*.jpg"))

extensions = (".jpg", ".jpeg", ".png")

predicted_images = [
    os.path.join("/content/", f)
    for f in os.listdir("/content/")
    if f.lower().endswith(extensions)
]

# Sort by last modified time (oldest → newest)
predicted_images.sort(key=os.path.getmtime)
latest_image = predicted_images[-1]

print("Opening:", latest_image,"\n")

DisplayImage(filename=latest_image,width=600)

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Run inference on an uploaded video and save it to the content folder
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

uploaded = files.upload()
uploaded_image_filename = list(uploaded.keys())[0]
input_video = "/content/" + uploadedvideofilename

results = model.predict(
    source=input_video,
    save=True,
    conf=0.25,
    name=f"/content",
    exist_ok=True
)

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Install FFMPEG
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

!apt-get -qq install ffmpeg

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Convert the the annotatd AVI into an MP4
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

!ffmpeg -i /content/gun.avi -vcodec libx264 -acodec aac /content/output.mp4 -y

In [ ]:
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# View the the converted MP4 file in an HTML Viewer
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

from IPython.display import HTML
from base64 import b64encode

mp4 = open("/content/output.mp4",'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<video width=640 controls>
      <source src="{data_url}" type="video/mp4">
</video>
""")